# NicoComic YOLOX-Nano panel training

Use a free Colab GPU runtime. This notebook trains only on UMD COMICS public-domain pages with MIT-licensed annotations. It does not upload private comics or annotations.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'

In [ ]:
!git clone -q https://github.com/zyuanming/NicoComic-Reader-Models.git
%cd /content/NicoComic-Reader-Models
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX
!git -C /content/YOLOX checkout -q 419778480ab6ec0590e5d3831b3afb3b46ab2aa3
!pip -q install -r scripts/requirements-training.txt
!pip -q install -e /content/YOLOX --no-build-isolation --no-deps

In [ ]:
!curl -fL --retry 3 -o /content/panels_annotations.zip https://obj.umiacs.umd.edu/comics/panels_annotations.zip
!python scripts/prepare_comics_coco.py /content/panels_annotations.zip /content/nicocomic-comics-coco

In [ ]:
!python /content/YOLOX/tools/train.py -f experiments/yolox_nano_panels.py -d 1 -b 16 --fp16 data_dir /content/nicocomic-comics-coco

In [ ]:
from pathlib import Path
checkpoint_dir = Path('YOLOX_outputs/yolox_nano_panels')
checkpoint = checkpoint_dir / 'best_ckpt.pth'
if not checkpoint.exists():
    checkpoint = checkpoint_dir / 'latest_ckpt.pth'
assert checkpoint.exists(), 'Training did not produce a checkpoint'
print(checkpoint)

In [ ]:
!python scripts/export_yolox_onnx.py experiments/yolox_nano_panels.py {checkpoint} /content/NicoComicPanelYOLOXNano416.onnx
from google.colab import files
files.download('/content/NicoComicPanelYOLOXNano416.onnx')